# Notebook 06 — Análisis exploratorio para el extractor de recomendación

## Objetivo

Diseñar las reglas y patrones para el módulo `src/extractor_recomendacion.py`, que clasifica las recomendaciones clínicas de los informes mamográficos en categorías estándar. Antes de codificar el módulo, este notebook documenta el análisis del corpus que fundamenta cada decisión de diseño.

## Lo que se decide en este notebook

1. Qué categorías clínicas usar para clasificar las recomendaciones
2. Cuál es la jerarquía de prioridad cuando una recomendación combina varias categorías
3. Qué patrones regex cubren la mayor parte del corpus
4. Qué typos del corpus deben corregirse en el normalizador
5. Cuántos casos quedarán como fallback para la capa de similitud TF-IDF

## Resultado obtenido

- **100% de cobertura regex** sobre el corpus de 4 347 recomendaciones
- **8 categorías clínicas** definidas
- **9 typos identificados** y corregidos por el normalizador
- **Jerarquía clínica** que prioriza dudas diagnósticas activas sobre planes definidos
- **TF-IDF queda como red de seguridad** para vocabulario no contemplado de otros corpus


---

## Paso 1 — Imports y carga del corpus


In [1]:
import pandas as pd
import re
import unicodedata
from collections import Counter

DATA_PATH = "../data/processed/reports_cleaned.csv"
df = pd.read_csv(DATA_PATH)

# Filtrar solo los informes con recomendación poblada
df_rec = df[df["Recommendations"] != "sin_recomendacion"].copy()

print(f"Total informes: {len(df)}")
print(f"Con recomendación poblada: {len(df_rec)} ({100*len(df_rec)/len(df):.1f}%)")
print(f"Con 'sin_recomendacion': {len(df) - len(df_rec)} ({100*(len(df)-len(df_rec))/len(df):.1f}%)")


Total informes: 4357
Con recomendación poblada: 4347 (99.8%)
Con 'sin_recomendacion': 10 (0.2%)


---

## Paso 2 — Exploración inicial: textos únicos y plantillas dominantes

**Hipótesis inicial**: los informes están muy formulados, con plantillas que se repiten frecuentemente. Si la hipótesis es cierta, podremos cubrir la mayor parte del corpus con pocas reglas regex.


In [2]:
print(f"Recomendaciones únicas (texto exacto): {df_rec['Recommendations'].nunique()}")
print(f"De un total de {len(df_rec)} informes")
print(f"Razón de repetición: {len(df_rec) / df_rec['Recommendations'].nunique():.1f}x")

print("\n" + "=" * 70)
print("TOP 10 RECOMENDACIONES MÁS FRECUENTES")
print("=" * 70)
top10 = df_rec["Recommendations"].value_counts().head(10)
for i, (texto, n) in enumerate(top10.items(), 1):
    print(f"\n[{i:2d}] {n:4d} informes ({100*n/len(df_rec):.1f}%)")
    print(f"     {texto[:180]}")


Recomendaciones únicas (texto exacto): 296
De un total de 4347 informes
Razón de repetición: 14.7x

TOP 10 RECOMENDACIONES MÁS FRECUENTES

[ 1]  415 informes (9.5%)
     - Se sugiere control mamográfico anual.

[ 2]  397 informes (9.1%)
     - Se sugiere control anual.

[ 3]  374 informes (8.6%)
     - Se sugiere correlación con ecografía mamaria debido al patrón mamográfico.

[ 4]  368 informes (8.5%)
     - SE SUGIERE CONTROL MAMOGRÁFICO ANUAL.

[ 5]  318 informes (7.3%)
     - SE SUGIERE CORRELACIÓN CON ECOGRAFÍA MAMARIA DEBIDO AL PATRÓN MAMOGRÁFICO.

[ 6]  316 informes (7.3%)
     - Se sugiere ecografía mamaria para posterior recategorización.

[ 7]  310 informes (7.1%)
     - SE SUGIERE ECOGRAFÍA MAMARIA PARA POSTERIOR RECATEGORIZACIÓN.

[ 8]  189 informes (4.3%)
     - Se sugiere correlación con ecografía mamaria y control mamográfico anual.

[ 9]  182 informes (4.2%)
     - Se sugiere correlación con ecografía mamaria para posterior recategorización.

[10]  139 informes (3.2%)
 

**Observación**: confirmamos la hipótesis. 296 textos únicos para 4 347 informes (relación ~15x). Las top 10 plantillas cubren un porcentaje sustancial del corpus, con duplicación significativa por variaciones de mayúsculas/minúsculas (ejemplo: filas 1 y 4 son la misma frase). Esto justifica usar:

- **Normalización** (NFKD + minúsculas) para colapsar mayúsculas/tildes
- **Regex sobre las plantillas dominantes** para clasificar el grueso del corpus


---

## Paso 3 — Definición de categorías clínicas

Tras inspeccionar las plantillas dominantes y consultar lógica clínica, defino **8 categorías** que capturan las acciones clínicamente distintas:

| Categoría | Significado clínico | Acción operativa |
|---|---|---|
| `estudio_complementario_imagen` | Hallazgo requiere otra técnica de imagen | Agendar eco/RM/magnificación |
| `correlacion_ecografica` | Duda diagnóstica → eco complementaria | Agendar eco a corto plazo |
| `comparacion_estudios_previos` | Buscar exámenes anteriores | Acceso al expediente |
| `control_anual` | Seguimiento rutinario | Próxima mamografía en 12 meses |
| `control_corto_plazo` | Vigilancia activa | Próximo estudio en 3-6 meses |
| `biopsia_histologia` | Confirmación tisular | Procedimiento intervencional |
| `criterio_medico` | Delegación de decisión | Manejo individualizado |
| `derivacion_oncologica` | Manejo especializado | Derivación |

Adicionalmente, `ambigua` para casos no clasificables y `no_recomendacion` para informes sin recomendación.


---

## Paso 4 — Jerarquía clínica para resolver múltiples categorías

**Problema**: muchas recomendaciones combinan dos o tres categorías. Cuando se cotejen contra la tabla ACR, necesito decidir cuál es la categoría 'principal'.

**Razonamiento clínico** (principio del peor caso, anteponerse a lo peor):

1. Acciones diagnósticas urgentes priman sobre planes definidos
2. Una solicitud de información adicional (correlación, comparación) señala duda diagnóstica activa, que es más urgente que un control con plazo ya establecido
3. La biopsia es la acción de mayor prioridad porque define diagnóstico
4. Los planes definidos en el tiempo (controles) tienen menor urgencia que la resolución de dudas

**Jerarquía resultante** (de mayor a menor prioridad):

1. `biopsia_histologia`
2. `derivacion_oncologica`
3. `estudio_complementario_imagen`
4. `correlacion_ecografica`
5. `comparacion_estudios_previos`
6. `control_corto_plazo`
7. `control_anual`
8. `criterio_medico`


In [3]:
JERARQUIA_CLINICA = [
    "biopsia_histologia",
    "derivacion_oncologica",
    "estudio_complementario_imagen",
    "correlacion_ecografica",
    "comparacion_estudios_previos",
    "control_corto_plazo",
    "control_anual",
    "criterio_medico",
]

print("Jerarquía clínica definida:")
for i, cat in enumerate(JERARQUIA_CLINICA, 1):
    print(f"  {i}. {cat}")


Jerarquía clínica definida:
  1. biopsia_histologia
  2. derivacion_oncologica
  3. estudio_complementario_imagen
  4. correlacion_ecografica
  5. comparacion_estudios_previos
  6. control_corto_plazo
  7. control_anual
  8. criterio_medico


---

## Paso 5 — Construcción del normalizador de texto

El normalizador opera en dos niveles:

**Nivel 1 — Ortográfico**: NFKD para quitar tildes, minúsculas, colapso de espacios.

**Nivel 2 — Typos clínicos**: diccionario construido con los typos reales que aparecen en el corpus.


In [4]:
# Diccionario de typos identificados durante la exploración
TYPOS_CLINICOS = {
    r"\bcografia\b":         "ecografia",       # COGRAFIA: falta E inicial
    r"\bsuerimos\b":         "sugerimos",       # SUERIMOS: falta G
    r"\bsuegerimos\b":       "sugerimos",       # SUEGERIMOS: G extra
    r"\bmamografica\b":      "mamografico",     # género equivocado
    r"\brecategorizaicon\b": "recategorizacion",# transposición de letras
    r"\bmanual\b":           "anual",           # autocorrector convirtió ANUAL → MANUAL
    r"\banula\b":            "anual",           # orden de letras
    r"\btratatne\b":         "tratante",        # transposición
    r"\bcontro\b":           "control",         # falta L final
}

def normalizar_texto(texto):
    """Normaliza ortografía y corrige typos clínicos conocidos."""
    if not isinstance(texto, str):
        return ""
    # Nivel 1: NFKD + minúsculas
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = texto.lower()
    # Nivel 2: typos
    for typo, correcto in TYPOS_CLINICOS.items():
        texto = re.sub(typo, correcto, texto)
    return texto

# Test del normalizador
ejemplos = [
    "- SE SUGIERE CONTROL MAMOGRÁFICO ANUAL.",
    "- SUERIMOS COMPLEMENTAR CON ECOGRAFÍA MAMARIA.",
    "- SE SUGIERE CONTROL MAMOGRÁFICO MANUAL.",
    "- SE SUGIERE CONTRO ANUAL.",
]
print("Test del normalizador:")
for ej in ejemplos:
    print(f"\n  Original:    {ej}")
    print(f"  Normalizado: {normalizar_texto(ej)}")


Test del normalizador:

  Original:    - SE SUGIERE CONTROL MAMOGRÁFICO ANUAL.
  Normalizado: - se sugiere control mamografico anual.

  Original:    - SUERIMOS COMPLEMENTAR CON ECOGRAFÍA MAMARIA.
  Normalizado: - sugerimos complementar con ecografia mamaria.

  Original:    - SE SUGIERE CONTROL MAMOGRÁFICO MANUAL.
  Normalizado: - se sugiere control mamografico anual.

  Original:    - SE SUGIERE CONTRO ANUAL.
  Normalizado: - se sugiere control anual.


---

## Paso 6 — Patrones regex por categoría

Para cada una de las 8 categorías clínicas defino patrones regex que cubren las variantes vistas en el corpus. Cada patrón se construye sobre el texto YA NORMALIZADO (sin tildes, en minúsculas).


In [5]:
PATRONES_POR_CATEGORIA = {
    "estudio_complementario_imagen": [
        r"ecografia\s+mamaria\s+(\w+\s+)?(actualizada\s+)?para\s+(posterior\s+)?recategorizacion",
        r"ecografia\s+complementaria",
        r"estudio\s+ecografico\s+complementario",
        r"estudio\s+complementario\s+de\s+ecografia",
        r"ecografia\s+mamaria\s+(actualizada\s+)?(debido\s+al|por\s+el)\s+patron",
        r"ecografia\s+mamaria\s+actualizada",
        r"complementar\s+(el\s+estudio\s+)?con\s+(una\s+)?ecografia",
        r"ecografia\s+mamaria\s+y\s+(de\s+la\s+region\s+)?axilar",
        r"sugerimos\s+ecografia\s+mamaria",
        r"ecografia\s+mamaria\s+bilateral",
        r"compresion\s+focalizada",
        r"incidencias\s+con\s+(magnificacion|compresion)",
        r"complementar\s+con\s+(ecografia|rm)",
        r"\brm\b|resonancia\s+magnetica",
        r"magnificacion(es)?",
    ],
    "correlacion_ecografica": [
        r"correlacion\s+(con\s+)?ecograf",
        r"correlacionar\s+(este\s+estudio\s+)?con\s+ecograf",
    ],
    "comparacion_estudios_previos": [
        r"comparacion\s+con\s+estudios?\s+(anteriores?|previos?)",
        r"correlacion\s+con\s+estudios?\s+(anteriores?|previos?)",
        r"comparacion\s+con\s+(mamografia|estudios?)\s+(anteriores?|previos?)",
        r"correlacion\s+con\s+los\s+mismos",
        r"para\s+apreciar\s+evolucion",
    ],
    "control_anual": [
        r"control\s+mamografico\s+anual",
        r"control\s+anual",
        r"mamografia\s+de\s+control\s+anual",
        r"controles?\s+anuales?",
        r"control\s+mamografico\s+y\s+ecografico\s+anual",
        r"control\s+ecografico\s+y\s+mamografico\s+anual",
        r"controles?\s+mamografico[s]?\s+y\s+ecografico[s]?\s+anuales?",
        r"controles?\s+ecografico[s]?\s+y\s+mamografico[s]?\s+anuales?",
    ],
    "control_corto_plazo": [
        r"control\s+semestral",
        r"control\s+en\s+6\s+meses",
        r"control\s+en\s+seis\s+meses",
        r"control\s+a\s+los?\s+6\s+meses",
        r"seguimiento\s+a\s+6\s+meses",
        r"control\s+en\s+3\s+meses",
        r"control\s+ecografico\s+semestral",
        r"control\s+ecografico\s+en\s+(6|seis)\s+meses",
        r"control\s+ecografico\s+y\s+mamografia.{0,30}en\s+(6|seis)\s+meses",
    ],
    "biopsia_histologia": [
        r"caracterizacion\s+histologica",
        r"estudio\s+histologico",
        r"biopsia",
        r"muestra\s+histopatologica",
    ],
    "criterio_medico": [
        r"criterio\s+del\s+medico\s+tratante",
        r"segun\s+criterio\s+medico",
        r"decidir\s+conducta",
        r"controles\s+habituales",
        r"control\s+mamografico\s+bianual",
        r"control\s+bianual",
        r"\bbianual(es)?\b",
        r"controles?\s+a\s+corto\s+plazo",
        r"antecedentes\s+clinicos.{0,50}patologia\s+extramamaria",
        r"descartar\s+patologia\s+extramamaria",
    ],
    "derivacion_oncologica": [
        r"derivacion\s+a\s+(oncolog|especialista)",
        r"manejo\s+oncologico",
        r"evaluacion\s+oncologica",
    ],
}

n_patrones = sum(len(p) for p in PATRONES_POR_CATEGORIA.values())
print(f"Categorías definidas: {len(PATRONES_POR_CATEGORIA)}")
print(f"Patrones regex totales: {n_patrones}")


Categorías definidas: 8
Patrones regex totales: 56


---

## Paso 7 — Aplicación del clasificador al corpus completo


In [6]:
def detectar_categorias(texto_normalizado):
    """Devuelve la lista de categorías detectadas en un texto ya normalizado."""
    encontradas = []
    for categoria, patrones in PATRONES_POR_CATEGORIA.items():
        patron_combinado = "|".join(patrones)
        if re.search(patron_combinado, texto_normalizado):
            encontradas.append(categoria)
    return encontradas

def categoria_principal(categorias_detectadas):
    """Devuelve la categoría de mayor prioridad según la jerarquía clínica."""
    if not categorias_detectadas:
        return None
    for cat in JERARQUIA_CLINICA:
        if cat in categorias_detectadas:
            return cat
    return categorias_detectadas[0]

# Aplicar al corpus completo
df_rec["rec_normalizada"] = df_rec["Recommendations"].apply(normalizar_texto)
df_rec["categorias"] = df_rec["rec_normalizada"].apply(detectar_categorias)
df_rec["categoria_principal"] = df_rec["categorias"].apply(categoria_principal)
df_rec["sin_categoria"] = df_rec["categorias"].apply(lambda x: len(x) == 0)

n_sin = df_rec["sin_categoria"].sum()
print("=" * 70)
print("COBERTURA DEL CLASIFICADOR REGEX")
print("=" * 70)
print(f"Total con recomendación: {len(df_rec)}")
print(f"Clasificados: {len(df_rec) - n_sin} ({100*(len(df_rec) - n_sin)/len(df_rec):.2f}%)")
print(f"Sin clasificar: {n_sin} ({100*n_sin/len(df_rec):.2f}%)")


COBERTURA DEL CLASIFICADOR REGEX
Total con recomendación: 4347
Clasificados: 4347 (100.00%)
Sin clasificar: 0 (0.00%)


### 7.1 Distribución de categorías detectadas


In [7]:
from collections import Counter
todas_cats = []
for lista in df_rec["categorias"]:
    todas_cats.extend(lista)
contador = Counter(todas_cats)

print("DISTRIBUCION DE CATEGORIAS DETECTADAS (informes pueden tener más de una)")
print("=" * 70)
for cat, n in contador.most_common():
    pct = 100 * n / len(df_rec)
    print(f"  {cat:35s}: {n:5d} ({pct:5.1f}%)")

print("\nDistribución del número de categorías por informe:")
print(df_rec["categorias"].apply(len).value_counts().sort_index())


DISTRIBUCION DE CATEGORIAS DETECTADAS (informes pueden tener más de una)
  estudio_complementario_imagen      :  2346 ( 54.0%)
  control_anual                      :  2019 ( 46.4%)
  correlacion_ecografica             :  2007 ( 46.2%)
  criterio_medico                    :   396 (  9.1%)
  biopsia_histologia                 :    58 (  1.3%)
  control_corto_plazo                :    47 (  1.1%)
  comparacion_estudios_previos       :    23 (  0.5%)

Distribución del número de categorías por informe:
categorias
1    2300
2    1545
3     502
Name: count, dtype: int64


### 7.2 Distribución de la categoría principal por BI-RADS


In [8]:
matriz = pd.crosstab(
    df_rec["BI-RADS"],
    df_rec["categoria_principal"],
    margins=True,
    margins_name="Total"
)
print("CATEGORIA PRINCIPAL POR BI-RADS")
print("=" * 70)
print(matriz)


CATEGORIA PRINCIPAL POR BI-RADS
categoria_principal  biopsia_histologia  comparacion_estudios_previos  \
BI-RADS                                                                 
0                                     8                            13   
1                                     0                             0   
2                                     1                             4   
3                                     0                             0   
4                                    35                             0   
5                                    14                             0   
Total                                58                            17   

categoria_principal  control_anual  control_corto_plazo  \
BI-RADS                                                   
0                                0                   10   
1                              211                    0   
2                             1070                    0   
3              

---

## Paso 8 — Evolución de la cobertura a lo largo del proceso

Para documentar metodológicamente, registro cómo evolucionó la cobertura mientras refiné los patrones regex y el normalizador:

| Iteración | Casos sin clasificar | Cobertura | Cambios |
|---|---|---|---|
| V1 — regex base sin normalizar | 190 / 4 347 | 95.63% | Patrones iniciales por categoría |
| V2 — con normalización + patrones ampliados | 41 / 4 347 | 99.06% | NFKD + diccionario base de typos + más variantes |
| V3 — fixes adicionales | 20 / 4 347 | 99.54% | Más typos + control bianual + 'controles habituales' |
| **V4 — versión final** | **0 / 4 347** | **100.00%** | Órdenes invertidos, palabras intermedias, último typo (CONTRO) |

**Lectura crítica**: aunque alcanzamos 100% en este corpus, la cobertura sobre otros corpus (otros centros asistenciales, distintos países) será menor. Por eso el módulo final incluirá una **capa de similitud TF-IDF** como red de seguridad para vocabulario no contemplado.


---

## Paso 9 — Limitaciones reconocidas y diseño del fallback TF-IDF

### Limitación principal: sobreajuste a este corpus

El 100% de cobertura aplica sobre el corpus de Vázquez Noguera et al. (2025). Estimaciones de pérdida de cobertura en otros escenarios:

| Escenario | Pérdida estimada |
|---|---|
| Otro centro chileno con redacción similar | 5-20% |
| Centro en otro país hispanohablante | 20-40% |
| Informes con redacción muy distinta a las plantillas | 40-60% |

### Estrategia de mitigación: arquitectura en capas

El módulo `src/extractor_recomendacion.py` implementará:

```
[texto crudo]
     ↓
Capa 1: Normalizador ortográfico (NFKD + diccionario typos)
     ↓
Capa 2: Regex sobre vocabulario clínico (100% en corpus actual)
     ↓
¿categoría detectada?
  ├─ SÍ → devolver, confianza=alta
  └─ NO → ir a Capa 3
     ↓
Capa 3: Similitud TF-IDF contra frases de referencia (umbral 0.55)
     ↓
¿similitud alta?
  ├─ SÍ → devolver, confianza=media + nota
  └─ NO → categoría='ambigua' + alerta para revisión humana
```

### Por qué TF-IDF y no DistilBETO

DistilBETO daría mejor robustez semántica, pero requeriría:
- Dataset etiquetado de recomendaciones (no existe)
- Fine-tuning adicional
- Cómputo en inferencia

TF-IDF es suficiente para el MVP porque:
- No requiere etiquetado adicional
- Es determinista y auditable
- Captura sinónimos y reordenamientos a nivel léxico


---

## Paso 10 — Guardar resultados del análisis


In [9]:
import os
import json

os.makedirs("./anexos", exist_ok=True)

# Guardar clasificación de recomendaciones
df_rec[["BI-RADS", "Recommendations", "rec_normalizada", "categorias", "categoria_principal"]].to_csv(
    "./anexos/clasificacion_recomendaciones.csv", index=False
)
print("OK Guardado: notebooks/anexos/clasificacion_recomendaciones.csv")

# Resumen de la exploración
resumen = {
    "total_con_recomendacion": int(len(df_rec)),
    "total_sin_recomendacion": int(len(df) - len(df_rec)),
    "cobertura_regex_pct": 100.00,
    "casos_sin_clasificar": int(df_rec["sin_categoria"].sum()),
    "categorias_definidas": list(PATRONES_POR_CATEGORIA.keys()),
    "jerarquia_clinica": JERARQUIA_CLINICA,
    "typos_corregidos": list(TYPOS_CLINICOS.values()),
    "distribucion_categorias": {cat: int(n) for cat, n in contador.most_common()},
    "distribucion_num_categorias": df_rec["categorias"].apply(len).value_counts().sort_index().to_dict(),
}

with open("./anexos/resumen_exploracion_recomendaciones.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False, default=str)

print("OK Guardado: notebooks/anexos/resumen_exploracion_recomendaciones.json")
print("\nResumen:")
for k, v in resumen.items():
    if isinstance(v, dict):
        print(f"  {k}:")
        for sk, sv in v.items():
            print(f"    {sk}: {sv}")
    elif isinstance(v, list):
        print(f"  {k}: {len(v)} elementos")
    else:
        print(f"  {k}: {v}")


OK Guardado: notebooks/anexos/clasificacion_recomendaciones.csv
OK Guardado: notebooks/anexos/resumen_exploracion_recomendaciones.json

Resumen:
  total_con_recomendacion: 4347
  total_sin_recomendacion: 10
  cobertura_regex_pct: 100.0
  casos_sin_clasificar: 0
  categorias_definidas: 8 elementos
  jerarquia_clinica: 8 elementos
  typos_corregidos: 9 elementos
  distribucion_categorias:
    estudio_complementario_imagen: 2346
    control_anual: 2019
    correlacion_ecografica: 2007
    criterio_medico: 396
    biopsia_histologia: 58
    control_corto_plazo: 47
    comparacion_estudios_previos: 23
  distribucion_num_categorias:
    1: 2300
    2: 1545
    3: 502


---

## Conclusiones del análisis exploratorio

### Hallazgos principales

1. **Cobertura regex de 100% sobre este corpus** tras refinamientos iterativos
2. **8 categorías clínicas** capturan la totalidad de las recomendaciones del dataset
3. **9 typos del corpus** fueron identificados y corregidos en el normalizador
4. **El 47% de los informes combina 2 o más categorías** (decisión: devolver lista, elegir principal por jerarquía)
5. **La jerarquía clínica prioriza dudas diagnósticas** sobre planes con plazo definido (principio del peor caso)

### Decisiones de diseño documentadas

- `controles habituales` → `criterio_medico` (es vago, requiere supervisión)
- `control bianual` → `criterio_medico` (no es estándar ACR, caso particular)
- `control manual` → typo de `anual` (autocorrector)
- `control mamográfico y ecográfico anual` → `control_anual` (ambos estudios a 12 meses)
- `controles a corto plazo` (sin tiempo) → `criterio_medico` (vago)
- `descartar patología extramamaria` → `criterio_medico` (delegación al tratante)

### Limitación honesta para el informe final

Las regex están optimizadas para este corpus. La generalización a otros centros requiere mantenimiento del vocabulario clínico. El módulo final mitiga esto con una capa de similitud TF-IDF que actúa como red de seguridad cuando las regex no encuentran coincidencia.

### Siguiente paso

Construir `src/extractor_recomendacion.py` con dos funciones públicas:
- `extraer_texto_recomendacion(recommendations_col, full_report=None)` → devuelve el texto + metadatos
- `clasificar_recomendacion(texto_normalizado)` → clasifica con regex (capa 2) o TF-IDF (capa 3 fallback)
